In [1]:

from pathlib import Path
import numpy as np
import pandas as pd

LAST_NAME = "Malitao"
STUDENT_ID = "TUPM-26-1135"

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
assert id_digits, "Student ID must contain digits."

SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
NAME_LENGTH = len(LAST_NAME)
RANDOM_SEED = int(id_digits) % (2**32)
verification_value = ID_SUM * NAME_LENGTH + SEED_DIGIT

np.set_printoptions(precision=4, suppress=True)

print("STUDENT CONFIGURATION")
print("---------------------")
print("Student:", LAST_NAME, STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Name Length:", NAME_LENGTH)
print("Random Seed:", RANDOM_SEED)
print("Verification Value:", verification_value)


STUDENT CONFIGURATION
---------------------
Student: Malitao TUPM-26-1135
Seed Digit: 5
ID Sum: 18
Name Length: 7
Random Seed: 261135
Verification Value: 131


In [2]:

# [CELL 2] DATASET LOADING AND INITIAL INSPECTION

DATA_DIR = Path("../data")

MEASUREMENT_FILE = DATA_DIR / "secom.data"
LABEL_FILE = DATA_DIR / "secom_labels.data"

# Load measurement matrix
measurements = np.loadtxt(
    MEASUREMENT_FILE,
    dtype=float
)

# Load labels
labels_raw = pd.read_csv(
    LABEL_FILE,
    sep=r"\s+",
    header=None,
    names=["Outcome", "Timestamp"]
)

# Verify matching record counts
assert len(measurements) == len(labels_raw), (
    "Measurement and label record counts do not match."
)

# Preserve original record identifiers
record_ids = np.arange(len(measurements))

# Create labeled DataFrame
labels = labels_raw["Outcome"].to_numpy()

secom_df = pd.DataFrame(
    measurements,
    index=record_ids
)

secom_df.index.name = "Record_ID"

secom_df["Outcome"] = labels

# Display dataset information
print("SECOM DATASET OVERVIEW")
print("----------------------")
print("Measurement Shape:", measurements.shape)
print("Label Shape:", labels_raw.shape)
print("Total Records:", len(record_ids))
print("Total Measurement Features:", measurements.shape[1])

print("\nOUTCOME DISTRIBUTION")
print("--------------------")
print(secom_df["Outcome"].value_counts().sort_index())

print("\nMISSING VALUE INSPECTION")
print("------------------------")
print("Total Missing Values:", np.isnan(measurements).sum())
print(
    "Features with Missing Values:",
    np.isnan(measurements).any(axis=0).sum()
)

print("\nFIRST FIVE RECORDS")
print("------------------")
display(secom_df.head())


SECOM DATASET OVERVIEW
----------------------
Measurement Shape: (1567, 590)
Label Shape: (1567, 2)
Total Records: 1567
Total Measurement Features: 590

OUTCOME DISTRIBUTION
--------------------
Outcome
-1    1463
 1     104
Name: count, dtype: int64

MISSING VALUE INSPECTION
------------------------
Total Missing Values: 41951
Features with Missing Values: 538

FIRST FIVE RECORDS
------------------


,0,1,2,3,4,5,6,7,8,9,...,581,582,583,584,585,586,587,588,589,Outcome
Record_ID,,,,,,,,,,,,,,,,,,,,,
0,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,0.0162,...,NaN,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN,-1
1,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,-0.0005,...,208.2045,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045,-1
2,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,0.0041,...,82.8602,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602,1
3,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,-0.0124,...,73.8432,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432,-1
4,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,-0.0031,...,NaN,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432,-1


In [3]:

# [CELL 3] FEATURE QUALITY IDENTIFICATION

# Count observed and missing values per feature
observed_counts = np.sum(~np.isnan(measurements), axis=0)
missing_counts = np.sum(np.isnan(measurements), axis=0)

missing_percentages = (
    missing_counts / len(measurements)
) * 100

# Calculate feature-wise observed minimum and maximum
feature_min = np.nanmin(measurements, axis=0)
feature_max = np.nanmax(measurements, axis=0)

# Identify features containing missing values
features_with_missing = missing_counts > 0

# Identify constant features among observed values
constant_features = (
    (observed_counts > 0) &
    (feature_min == feature_max)
)

# Identify features with no observed values
all_missing_features = observed_counts == 0

# Identify usable features
usable_features = np.where(
    (observed_counts > 0) &
    (~constant_features)
)[0]

print("FEATURE QUALITY ANALYSIS")
print("------------------------")
print("Total Features:", measurements.shape[1])
print("Features with Missing Values:", features_with_missing.sum())
print("Constant Features:", constant_features.sum())
print("Features with No Observed Values:", all_missing_features.sum())
print("Usable Features:", len(usable_features))

print("\nHIGHEST MISSING PERCENTAGE")
print("-------------------------")
print(f"{missing_percentages.max():.4f}%")

print("\nUSABLE FEATURE INDICES")
print("----------------------")
print(usable_features)


FEATURE QUALITY ANALYSIS
------------------------
Total Features: 590
Features with Missing Values: 538
Constant Features: 116
Features with No Observed Values: 0
Usable Features: 474

HIGHEST MISSING PERCENTAGE
-------------------------
91.1934%

USABLE FEATURE INDICES
----------------------
[  0   1   2   3   4   6   7   8   9  10  11  12  14  15  16  17  18  19
  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36  37
  38  39  40  41  43  44  45  46  47  48  50  51  53  54  55  56  57  58
  59  60  61  62  63  64  65  66  67  68  70  71  72  73  74  75  76  77
  78  79  80  81  82  83  84  85  86  87  88  89  90  91  92  93  94  95
  96  98  99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114
 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132
 133 134 135 136 137 138 139 140 142 143 144 145 146 147 148 150 151 152
 153 154 155 156 157 158 159 160 161 162 163 164 165 166 167 168 169 170
 171 172 173 174 175 176 177 180 181 182 183 184 

In [4]:

# [CELL 4] REPRODUCIBLE FEATURE SELECTION

SELECTED_FEATURE_COUNT = 5 + (SEED_DIGIT % 3)

assert len(usable_features) >= SELECTED_FEATURE_COUNT, (
    "Not enough usable features."
)

rng = np.random.default_rng(RANDOM_SEED)

selected_features = np.sort(
    rng.choice(
        usable_features,
        size=SELECTED_FEATURE_COUNT,
        replace=False
    )
)

print("FEATURE SELECTION")
print("-----------------")
print("Required Features:", SELECTED_FEATURE_COUNT)
print("Selected Feature Indices:", selected_features)
print("Number of Selected Features:", len(selected_features))


FEATURE SELECTION
-----------------
Required Features: 7
Selected Feature Indices: [  3  71 157 168 297 431 551]
Number of Selected Features: 7


In [5]:

# [CELL 5] FEATURE-WISE STATISTICAL ANALYSIS

selected_data = measurements[:, selected_features]

feature_means = np.nanmean(selected_data, axis=0)
feature_medians = np.nanmedian(selected_data, axis=0)
feature_std = np.nanstd(selected_data, axis=0, ddof=1)

selected_missing_percentages = (
    np.isnan(selected_data).mean(axis=0) * 100
)

statistics_df = pd.DataFrame({
    "Feature_Index": selected_features,
    "Mean": feature_means,
    "Median": feature_medians,
    "Sample_Std": feature_std,
    "Missing_Percentage": selected_missing_percentages
})

print("SELECTED FEATURE STATISTICS")
print("----------------------------")
display(statistics_df.round(4))


SELECTED FEATURE STATISTICS
----------------------------


,Feature_Index,Mean,Median,Sample_Std,Missing_Percentage
0,3,1396.3766,1285.2144,441.6916,0.8934
1,71,104.3290,102.6043,31.6519,0.3829
2,157,0.0471,0.0354,0.0395,91.1934
3,168,0.1244,0.1190,0.0476,0.1276
4,297,2342.8270,820.0988,3226.9243,0.1276
5,431,22.3583,16.3810,36.3954,0.1276
6,551,1.2307,1.1500,1.3611,16.5922


In [6]:

# [CELL 6] ROBUST STANDARDIZATION USING MAD

MAD_SCALE = 0.6745

robust_z_scores = np.full(
    selected_data.shape,
    np.nan,
    dtype=float
)

feature_mads = np.zeros(len(selected_features))

for j in range(len(selected_features)):

    column = selected_data[:, j]
    valid_values = column[~np.isnan(column)]

    median_value = np.median(valid_values)

    mad_value = np.median(
        np.abs(valid_values - median_value)
    )

    feature_mads[j] = mad_value

    if mad_value > 0:
        robust_z_scores[:, j] = (
            MAD_SCALE * (column - median_value) / mad_value
        )

    else:
        # Safely handle zero MAD
        robust_z_scores[:, j] = np.where(
            np.isnan(column),
            np.nan,
            0.0
        )

print("ROBUST STANDARDIZATION")
print("----------------------")
print("MAD Values:", np.round(feature_mads, 4))
print("Robust Z-Score Shape:", robust_z_scores.shape)
print("Zero MAD Features:", np.sum(feature_mads == 0))


ROBUST STANDARDIZATION
----------------------
MAD Values: [245.9853  13.8556   0.0107   0.03   754.8589   5.3298   0.27  ]
Robust Z-Score Shape: (1567, 7)
Zero MAD Features: 0


In [7]:

# [CELL 7] OUTLIER DETECTION

OUTLIER_THRESHOLD = 3.5

# Identify flagged observations
flagged_mask = (
    np.abs(robust_z_scores) > OUTLIER_THRESHOLD
)

# Count flagged observations per feature
flagged_counts = np.sum(flagged_mask, axis=0)

# Calculate flagged percentages
flagged_percentages = (
    flagged_counts / observed_counts[selected_features]
) * 100

# Count unique records with at least one flagged measurement
records_with_outliers = np.any(flagged_mask, axis=1)
total_flagged_observations = np.sum(flagged_mask)
unique_flagged_records = np.sum(records_with_outliers)

# Update statistics DataFrame
statistics_df["MAD"] = feature_mads
statistics_df["Flagged_Count"] = flagged_counts
statistics_df["Flagged_Percentage"] = flagged_percentages

statistics_df = statistics_df.sort_values(
    by="Flagged_Percentage",
    ascending=False
).reset_index(drop=True)

print("OUTLIER DETECTION RESULTS")
print("-------------------------")
print("Outlier Threshold:", OUTLIER_THRESHOLD)
print("Total Flagged Observations:", total_flagged_observations)
print("Unique Records with Outliers:", unique_flagged_records)

print("\nFEATURES RANKED BY FLAGGED PERCENTAGE")
print("------------------------------------")
display(statistics_df.round(4))


OUTLIER DETECTION RESULTS
-------------------------
Outlier Threshold: 3.5
Total Flagged Observations: 527
Unique Records with Outliers: 478

FEATURES RANKED BY FLAGGED PERCENTAGE
------------------------------------


,Feature_Index,Mean,Median,Sample_Std,Missing_Percentage,MAD,Flagged_Count,Flagged_Percentage
0,297,2342.8270,820.0988,3226.9243,0.1276,754.8589,307,19.6166
1,157,0.0471,0.0354,0.0395,91.1934,0.0107,12,8.6957
2,71,104.3290,102.6043,31.6519,0.3829,13.8556,84,5.3812
3,431,22.3583,16.3810,36.3954,0.1276,5.3298,69,4.4089
4,3,1396.3766,1285.2144,441.6916,0.8934,245.9853,36,2.3181
5,551,1.2307,1.1500,1.3611,16.5922,0.2700,11,0.8416
6,168,0.1244,0.1190,0.0476,0.1276,0.0300,8,0.5112


In [8]:

# [CELL 8] EXPORT MEASUREMENT QUALITY SUMMARY

OUTPUT_DIR = Path(".")

measurement_quality = statistics_df[
    [
        "Feature_Index",
        "Mean",
        "Median",
        "Sample_Std",
        "Missing_Percentage",
        "MAD",
        "Flagged_Count",
        "Flagged_Percentage"
    ]
].copy()

measurement_quality.to_csv(
    OUTPUT_DIR / "measurement_quality.csv",
    index=False
)

# Identify assessment values
highest_missing_feature = selected_features[
    np.argmax(selected_missing_percentages)
]

highest_flagged_feature = statistics_df.iloc[0]["Feature_Index"]

print("MEASUREMENT QUALITY EXPORT")
print("--------------------------")
print("Selected Features:", len(selected_features))
print("Highest Missing Percentage:", f"{selected_missing_percentages.max():.4f}%")
print("Feature with Highest Missing Percentage:", highest_missing_feature)
print("Feature with Highest Flagged Percentage:", highest_flagged_feature)
print("Total Flagged Observations:", total_flagged_observations)
print("Vectorized Results Verified:", measurement_quality.shape[0] == len(selected_features))
print("Exported File:", OUTPUT_DIR / "measurement_quality.csv")
print("File Exists:", (OUTPUT_DIR / "measurement_quality.csv").exists())


MEASUREMENT QUALITY EXPORT
--------------------------
Selected Features: 7
Highest Missing Percentage: 91.1934%
Feature with Highest Missing Percentage: 157
Feature with Highest Flagged Percentage: 297.0
Total Flagged Observations: 527
Vectorized Results Verified: True
Exported File: measurement_quality.csv
File Exists: True


In [9]:
# EXERCISE 2 MOBILE DEVICE USAGE PROFILLING
# [CELL 9] PREPARE LABELED DATAFRAME

feature_columns = [f"Feature_{i}" for i in range(measurements.shape[1])]

labeled_df = pd.DataFrame(
    measurements,
    columns=feature_columns,
    index=record_ids
)

labeled_df.index.name = "Record_ID"
labeled_df["Outcome"] = labels

print("LABELED DATAFRAME")
print("-----------------")
print("Shape:", labeled_df.shape)
print("Original Record IDs Preserved:", labeled_df.index.equals(pd.Index(record_ids)))
print("Outcome Distribution:")
print(labeled_df["Outcome"].value_counts().sort_index())

display(labeled_df.head())


LABELED DATAFRAME
-----------------
Shape: (1567, 591)
Original Record IDs Preserved: True
Outcome Distribution:
Outcome
-1    1463
 1     104
Name: count, dtype: int64


,Feature_0,Feature_1,Feature_2,Feature_3,Feature_4,Feature_5,Feature_6,Feature_7,Feature_8,Feature_9,...,Feature_581,Feature_582,Feature_583,Feature_584,Feature_585,Feature_586,Feature_587,Feature_588,Feature_589,Outcome
Record_ID,,,,,,,,,,,,,,,,,,,,,
0,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,0.0162,...,NaN,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN,-1
1,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,-0.0005,...,208.2045,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045,-1
2,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,0.0041,...,82.8602,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602,1
3,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,-0.0124,...,73.8432,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432,-1
4,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,-0.0031,...,NaN,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432,-1


In [10]:

# [CELL 10] REPRODUCIBLE BALANCED SAMPLING

SAMPLE_PER_GROUP = min(
    20 + ID_SUM,
    (labels == -1).sum(),
    (labels == 1).sum()
)

rng_sample = np.random.default_rng(RANDOM_SEED)

pass_ids = labeled_df.index[
    labeled_df["Outcome"] == -1
].to_numpy()

fail_ids = labeled_df.index[
    labeled_df["Outcome"] == 1
].to_numpy()

selected_pass_ids = rng_sample.choice(
    pass_ids,
    size=SAMPLE_PER_GROUP,
    replace=False
)

selected_fail_ids = rng_sample.choice(
    fail_ids,
    size=SAMPLE_PER_GROUP,
    replace=False
)

balanced_ids = np.concatenate([
    selected_pass_ids,
    selected_fail_ids
])

balanced_df = labeled_df.loc[balanced_ids].copy()

print("BALANCED SAMPLING RESULTS")
print("-------------------------")
print("Sample Per Outcome:", SAMPLE_PER_GROUP)
print("Total Balanced Sample:", len(balanced_df))
print("Pass Records:", (balanced_df["Outcome"] == -1).sum())
print("Fail Records:", (balanced_df["Outcome"] == 1).sum())
print("Unique Record IDs:", balanced_df.index.nunique())


BALANCED SAMPLING RESULTS
-------------------------
Sample Per Outcome: 38
Total Balanced Sample: 76
Pass Records: 38
Fail Records: 38
Unique Record IDs: 76


In [11]:

# [CELL 11] REPRODUCIBLE FEATURE SELECTION

EX2_FEATURE_COUNT = 5

rng_ex2 = np.random.default_rng(RANDOM_SEED + 1)

ex2_selected_features = np.sort(
    rng_ex2.choice(
        usable_features,
        size=EX2_FEATURE_COUNT,
        replace=False
    )
)

ex2_columns = [
    f"Feature_{i}" for i in ex2_selected_features
]

print("EXERCISE 2 FEATURE SELECTION")
print("---------------------------")
print("Required Features:", EX2_FEATURE_COUNT)
print("Selected Feature Indices:", ex2_selected_features)
print("Selected Feature Columns:", ex2_columns)


EXERCISE 2 FEATURE SELECTION
---------------------------
Required Features: 5
Selected Feature Indices: [ 81 269 423 476 551]
Selected Feature Columns: ['Feature_81', 'Feature_269', 'Feature_423', 'Feature_476', 'Feature_551']


In [12]:

# [CELL 12] MEDIAN IMPUTATION USING FULL DATASET

full_feature_medians = np.nanmedian(
    measurements[:, ex2_selected_features],
    axis=0
)

balanced_features = balanced_df[ex2_columns].copy()

for column, median_value in zip(
    ex2_columns,
    full_feature_medians
):
    balanced_features[column] = (
        balanced_features[column].fillna(median_value)
    )

balanced_imputed_df = balanced_features.copy()
balanced_imputed_df["Outcome"] = balanced_df["Outcome"]

print("MEDIAN IMPUTATION")
print("-----------------")
print("Full Dataset Medians:", np.round(full_feature_medians, 4))
print("Missing Values Before Imputation:",
      balanced_features.isna().sum().sum())
print("Missing Values After Imputation:",
      balanced_imputed_df[ex2_columns].isna().sum().sum())
print("Imputation Complete:",
      balanced_imputed_df[ex2_columns].notna().all().all())


MEDIAN IMPUTATION
-----------------
Full Dataset Medians: [-0.0196  3.7035 69.9054 15.9738  1.15  ]
Missing Values Before Imputation: 0
Missing Values After Imputation: 0
Imputation Complete: True


In [13]:

# [CELL 13] GROUPED STATISTICAL ANALYSIS

grouped_stats = balanced_imputed_df.groupby("Outcome")[ex2_columns].agg(
    ["count", "mean", "median", "std"]
)

print("GROUPED STATISTICAL RESULTS")
print("---------------------------")
print("Outcome Labels: -1 = Pass, 1 = Fail")
display(grouped_stats.round(4))


GROUPED STATISTICAL RESULTS
---------------------------
Outcome Labels: -1 = Pass, 1 = Fail


Feature_81                         Feature_269                  \
             count    mean  median     std       count    mean  median   
Outcome                                                                  
-1              38 -0.0209 -0.0191  0.0181          38  3.6968  3.6412   
 1              38 -0.0182 -0.0183  0.0192          38  3.6779  3.5752   

                Feature_423                            Feature_476           \
            std       count     mean   median      std       count     mean   
Outcome                                                                       
-1       1.4431          38  69.9860  60.7433  32.4144          38  20.6632   
 1       0.9972          38  90.7543  82.6886  37.1771          38  22.8872   

                          Feature_551                         
          median      std       count    mean median     std  
Outcome                                                       
-1       15.9448  12.3295          38  1.2155   1.15  0.3287  
 1       16.8642  24.6681          38  1.1547   1.15  0.4560

In [14]:

# [CELL 14] PASS-FAIL FEATURE COMPARISON

pass_stats = grouped_stats.loc[-1]
fail_stats = grouped_stats.loc[1]

comparison_rows = []

for column, feature_index in zip(ex2_columns, ex2_selected_features):

    pass_count = int(pass_stats[(column, "count")])
    fail_count = int(fail_stats[(column, "count")])

    pass_mean = pass_stats[(column, "mean")]
    fail_mean = fail_stats[(column, "mean")]

    pass_median = pass_stats[(column, "median")]
    fail_median = fail_stats[(column, "median")]

    pass_std = pass_stats[(column, "std")]
    fail_std = fail_stats[(column, "std")]

    absolute_mean_difference = abs(fail_mean - pass_mean)

    pooled_variance = (
        ((pass_count - 1) * pass_std**2) +
        ((fail_count - 1) * fail_std**2)
    ) / (pass_count + fail_count - 2)

    pooled_std = np.sqrt(pooled_variance)

    if pooled_std > 0:
        smd = absolute_mean_difference / pooled_std
    elif absolute_mean_difference == 0:
        smd = 0.0
    else:
        smd = np.nan

    comparison_rows.append({
        "Feature_Index": feature_index,
        "Pass_Count": pass_count,
        "Fail_Count": fail_count,
        "Pass_Mean": pass_mean,
        "Fail_Mean": fail_mean,
        "Pass_Median": pass_median,
        "Fail_Median": fail_median,
        "Pass_Std": pass_std,
        "Fail_Std": fail_std,
        "Absolute_Mean_Difference": absolute_mean_difference,
        "Pooled_Std": pooled_std,
        "Standardized_Mean_Difference": smd
    })

comparison_df = pd.DataFrame(comparison_rows)

comparison_df = comparison_df.sort_values(
    by="Standardized_Mean_Difference",
    ascending=False,
    na_position="last"
).reset_index(drop=True)

print("PASS-FAIL FEATURE COMPARISON")
print("----------------------------")
display(comparison_df.round(4))


PASS-FAIL FEATURE COMPARISON
----------------------------


,Feature_Index,Pass_Count,Fail_Count,Pass_Mean,Fail_Mean,Pass_Median,Fail_Median,Pass_Std,Fail_Std,Absolute_Mean_Difference,Pooled_Std,Standardized_Mean_Difference
0,423,38,38,69.9860,90.7543,60.7433,82.6886,32.4144,37.1771,20.7683,34.8771,0.5955
1,551,38,38,1.2155,1.1547,1.1500,1.1500,0.3287,0.4560,0.0608,0.3975,0.1529
2,81,38,38,-0.0209,-0.0182,-0.0191,-0.0183,0.0181,0.0192,0.0027,0.0186,0.1475
3,476,38,38,20.6632,22.8872,15.9448,16.8642,12.3295,24.6681,2.2240,19.5004,0.1140
4,269,38,38,3.6968,3.6779,3.6412,3.5752,1.4431,0.9972,0.0189,1.2404,0.0152


In [15]:

# [CELL 15] NUMPY-PANDAS VERIFICATION

numpy_results_match = True

for outcome in [-1, 1]:

    group_data = balanced_imputed_df.loc[
        balanced_imputed_df["Outcome"] == outcome,
        ex2_columns
    ].to_numpy(dtype=float)

    pandas_group = grouped_stats.loc[outcome]

    numpy_means = np.mean(group_data, axis=0)
    numpy_medians = np.median(group_data, axis=0)
    numpy_stds = np.std(group_data, axis=0, ddof=1)

    pandas_means = pandas_group.loc[
        :, "mean"
    ].to_numpy(dtype=float)

    pandas_medians = pandas_group.loc[
        :, "median"
    ].to_numpy(dtype=float)

    pandas_stds = pandas_group.loc[
        :, "std"
    ].to_numpy(dtype=float)

    means_match = np.allclose(numpy_means, pandas_means)
    medians_match = np.allclose(numpy_medians, pandas_medians)
    stds_match = np.allclose(numpy_stds, pandas_stds)

    numpy_results_match &= (
        means_match and medians_match and stds_match
    )

    print(f"\nOutcome {outcome}:")
    print("Means Match:", means_match)
    print("Medians Match:", medians_match)
    print("Sample Standard Deviations Match:", stds_match)

print("\nOVERALL NUMPY-PANDAS VERIFICATION")
print("--------------------------------")
print("All Statistics Match:", numpy_results_match)



Outcome -1:
Means Match: True
Medians Match: True
Sample Standard Deviations Match: True

Outcome 1:
Means Match: True
Medians Match: True
Sample Standard Deviations Match: True

OVERALL NUMPY-PANDAS VERIFICATION
--------------------------------
All Statistics Match: True


In [16]:

# [CELL 16] EXPORT EXERCISE 2 FILES

# Preserve original Record_ID in the exported sample
balanced_export = balanced_imputed_df.copy()
balanced_export.insert(
    0,
    "Record_ID",
    balanced_df.index.to_numpy()
)

# Export balanced sample
balanced_export.to_csv(
    OUTPUT_DIR / "balanced_sample.csv",
    index=False
)

# Export feature comparison
comparison_df.to_csv(
    OUTPUT_DIR / "feature_comparison.csv",
    index=False
)

print("EXERCISE 2 EXPORT")
print("-----------------")
print("Balanced Sample Exists:",
      (OUTPUT_DIR / "balanced_sample.csv").exists())
print("Feature Comparison Exists:",
      (OUTPUT_DIR / "feature_comparison.csv").exists())

print("Balanced Sample Rows:", len(balanced_export))
print("Feature Comparison Rows:", len(comparison_df))
print("NumPy-Pandas Results Match:", numpy_results_match)


EXERCISE 2 EXPORT
-----------------
Balanced Sample Exists: True
Feature Comparison Exists: True
Balanced Sample Rows: 76
Feature Comparison Rows: 5
NumPy-Pandas Results Match: True


In [17]:
# EXERCISE 3

# [CELL 17] SELECT FEATURES FOR ROBUSTNESS ANALYSIS

pass_mask = labels == -1
fail_mask = labels == 1

eligible_features = []

for feature_index in usable_features:

    pass_observed = np.sum(
        ~np.isnan(measurements[pass_mask, feature_index])
    )

    fail_observed = np.sum(
        ~np.isnan(measurements[fail_mask, feature_index])
    )

    if pass_observed >= 10 and fail_observed >= 10:
        eligible_features.append(feature_index)

eligible_features = np.array(eligible_features)

EX3_FEATURE_COUNT = 5

assert len(eligible_features) >= EX3_FEATURE_COUNT, (
    "Not enough eligible features."
)

rng_ex3 = np.random.default_rng(RANDOM_SEED + 2)

ex3_selected_features = np.sort(
    rng_ex3.choice(
        eligible_features,
        size=EX3_FEATURE_COUNT,
        replace=False
    )
)

print("ROBUSTNESS FEATURE SELECTION")
print("----------------------------")
print("Eligible Features:", len(eligible_features))
print("Required Features:", EX3_FEATURE_COUNT)
print("Selected Feature Indices:", ex3_selected_features)

for feature_index in ex3_selected_features:

    pass_observed = np.sum(
        ~np.isnan(measurements[pass_mask, feature_index])
    )

    fail_observed = np.sum(
        ~np.isnan(measurements[fail_mask, feature_index])
    )

    print(
        f"Feature {feature_index}: "
        f"Pass = {pass_observed}, Fail = {fail_observed}"
    )


ROBUSTNESS FEATURE SELECTION
----------------------------
Eligible Features: 470
Required Features: 5
Selected Feature Indices: [162 417 444 473 522]
Feature 162: Pass = 1462, Fail = 103
Feature 417: Pass = 1461, Fail = 104
Feature 444: Pass = 1462, Fail = 104
Feature 473: Pass = 1456, Fail = 104
Feature 522: Pass = 1463, Fail = 104


In [18]:

# [CELL 18] PREPARE DATA FOR BOTH STRATEGIES

ex3_columns = [
    f"Feature_{i}" for i in ex3_selected_features
]

# Separate original measurements by outcome
pass_data = measurements[pass_mask][:, ex3_selected_features]
fail_data = measurements[fail_mask][:, ex3_selected_features]

# Calculate medians from the full dataset
ex3_full_medians = np.nanmedian(
    measurements[:, ex3_selected_features],
    axis=0
)

# Apply full-dataset median imputation
pass_imputed = np.where(
    np.isnan(pass_data),
    ex3_full_medians,
    pass_data
)

fail_imputed = np.where(
    np.isnan(fail_data),
    ex3_full_medians,
    fail_data
)

print("ROBUSTNESS DATA PREPARATION")
print("---------------------------")
print("Selected Features:", ex3_selected_features)
print("Full Dataset Medians:", np.round(ex3_full_medians, 4))

print("\nAVAILABLE-CASE OBSERVATIONS")
for j, feature_index in enumerate(ex3_selected_features):
    print(
        f"Feature {feature_index}: "
        f"Pass = {np.sum(~np.isnan(pass_data[:, j]))}, "
        f"Fail = {np.sum(~np.isnan(fail_data[:, j]))}"
    )

print("\nIMPUTATION VERIFICATION")
print("Pass Missing Values:", np.isnan(pass_imputed).sum())
print("Fail Missing Values:", np.isnan(fail_imputed).sum())
print(
    "Imputation Complete:",
    not np.isnan(pass_imputed).any()
    and not np.isnan(fail_imputed).any()
)


ROBUSTNESS DATA PREPARATION
---------------------------
Selected Features: [162 417 444 473 522]
Full Dataset Medians: [1784.        7.3956    0.9027   34.2468   13.7426]

AVAILABLE-CASE OBSERVATIONS
Feature 162: Pass = 1462, Fail = 103
Feature 417: Pass = 1461, Fail = 104
Feature 444: Pass = 1462, Fail = 104
Feature 473: Pass = 1456, Fail = 104
Feature 522: Pass = 1463, Fail = 104

IMPUTATION VERIFICATION
Pass Missing Values: 0
Fail Missing Values: 0
Imputation Complete: True


In [19]:

# [CELL 19] OBSERVED MEAN DIFFERENCES

available_case_differences = []
imputed_differences = []

for j in range(EX3_FEATURE_COUNT):

    pass_observed = pass_data[:, j]
    fail_observed = fail_data[:, j]

    pass_observed = pass_observed[~np.isnan(pass_observed)]
    fail_observed = fail_observed[~np.isnan(fail_observed)]

    available_difference = (
        np.mean(fail_observed) - np.mean(pass_observed)
    )

    imputed_difference = (
        np.mean(fail_imputed[:, j]) -
        np.mean(pass_imputed[:, j])
    )

    available_case_differences.append(available_difference)
    imputed_differences.append(imputed_difference)

strategy_comparison = pd.DataFrame({
    "Feature_Index": ex3_selected_features,
    "Available_Case_Difference": available_case_differences,
    "Median_Imputed_Difference": imputed_differences
})

print("OBSERVED MEAN DIFFERENCES")
print("-------------------------")
display(strategy_comparison.round(6))


OBSERVED MEAN DIFFERENCES
-------------------------


,Feature_Index,Available_Case_Difference,Median_Imputed_Difference
0,162,-154.061108,-179.583384
1,417,-0.693796,-0.692646
2,444,-0.004192,-0.004197
3,473,0.855168,0.879680
4,522,0.390212,0.390212


In [20]:

# [CELL 20] STRATIFIED BOOTSTRAP FUNCTION

BOOTSTRAP_REPETITIONS = 200 + (10 * SEED_DIGIT)
CONFIDENCE_LEVEL = 95

def stratified_bootstrap(
    pass_values,
    fail_values,
    repetitions,
    seed
):
    """
    Independently resample Pass and Fail observations
    with replacement and calculate Fail Mean - Pass Mean.
    """

    rng = np.random.default_rng(seed)

    bootstrap_differences = np.empty(repetitions)

    for i in range(repetitions):

        pass_sample = rng.choice(
            pass_values,
            size=len(pass_values),
            replace=True
        )

        fail_sample = rng.choice(
            fail_values,
            size=len(fail_values),
            replace=True
        )

        bootstrap_differences[i] = (
            np.mean(fail_sample) - np.mean(pass_sample)
        )

    bootstrap_mean = np.mean(bootstrap_differences)

    lower_bound = np.percentile(
        bootstrap_differences, 2.5
    )

    upper_bound = np.percentile(
        bootstrap_differences, 97.5
    )

    return {
        "Bootstrap_Mean_Difference": bootstrap_mean,
        "CI_Lower": lower_bound,
        "CI_Upper": upper_bound
    }


print("BOOTSTRAP CONFIGURATION")
print("----------------------")
print("Bootstrap Repetitions:", BOOTSTRAP_REPETITIONS)
print("Confidence Level:", CONFIDENCE_LEVEL, "%")
print("Resampling Method: Stratified")
print("Sampling: With Replacement")


BOOTSTRAP CONFIGURATION
----------------------
Bootstrap Repetitions: 250
Confidence Level: 95 %
Resampling Method: Stratified
Sampling: With Replacement


In [21]:

# [CELL 21] BOOTSTRAP ANALYSIS

bootstrap_results = []

for j, feature_index in enumerate(ex3_selected_features):

    # Available-case observations
    pass_available = pass_data[:, j]
    fail_available = fail_data[:, j]

    pass_available = pass_available[~np.isnan(pass_available)]
    fail_available = fail_available[~np.isnan(fail_available)]

    available_result = stratified_bootstrap(
        pass_available,
        fail_available,
        BOOTSTRAP_REPETITIONS,
        RANDOM_SEED + int(feature_index) * 2
    )

    # Median-imputed observations
    imputed_result = stratified_bootstrap(
        pass_imputed[:, j],
        fail_imputed[:, j],
        BOOTSTRAP_REPETITIONS,
        RANDOM_SEED + int(feature_index) * 2 + 1
    )

    bootstrap_results.append({
        "Feature_Index": feature_index,

        "Available_Case_Difference":
            available_case_differences[j],

        "Available_Case_Bootstrap_Mean":
            available_result["Bootstrap_Mean_Difference"],

        "Available_Case_CI_Lower":
            available_result["CI_Lower"],

        "Available_Case_CI_Upper":
            available_result["CI_Upper"],

        "Imputed_Difference":
            imputed_differences[j],

        "Imputed_Bootstrap_Mean":
            imputed_result["Bootstrap_Mean_Difference"],

        "Imputed_CI_Lower":
            imputed_result["CI_Lower"],

        "Imputed_CI_Upper":
            imputed_result["CI_Upper"]
    })

bootstrap_df = pd.DataFrame(bootstrap_results)

print("STRATIFIED BOOTSTRAP RESULTS")
print("----------------------------")
print("Total Features:", len(bootstrap_df))
print("Repetitions Per Strategy:", BOOTSTRAP_REPETITIONS)
display(bootstrap_df.round(6))


STRATIFIED BOOTSTRAP RESULTS
----------------------------
Total Features: 5
Repetitions Per Strategy: 250


,Feature_Index,Available_Case_Difference,Available_Case_Bootstrap_Mean,Available_Case_CI_Lower,Available_Case_CI_Upper,Imputed_Difference,Imputed_Bootstrap_Mean,Imputed_CI_Lower,Imputed_CI_Upper
0,162,-154.061108,-154.082686,-1114.133710,920.525405,-179.583384,-233.257260,-1357.082053,1016.906532
1,417,-0.693796,-0.735483,-1.409080,0.019194,-0.692646,-0.690523,-1.392820,0.181072
2,444,-0.004192,-0.003293,-0.027278,0.022741,-0.004197,-0.004911,-0.035354,0.025285
3,473,0.855168,0.979109,-3.803769,6.133184,0.879680,0.863539,-3.605700,5.440694
4,522,0.390212,0.385296,-0.788383,1.574159,0.390212,0.416497,-0.777834,1.915534


In [22]:

# [CELL 22] BOOTSTRAP REPRODUCIBILITY VERIFICATION

repeated_results = []

for j, feature_index in enumerate(ex3_selected_features):

    pass_available = pass_data[:, j]
    fail_available = fail_data[:, j]

    pass_available = pass_available[~np.isnan(pass_available)]
    fail_available = fail_available[~np.isnan(fail_available)]

    # Repeat available-case bootstrap
    available_repeat = stratified_bootstrap(
        pass_available,
        fail_available,
        BOOTSTRAP_REPETITIONS,
        RANDOM_SEED + int(feature_index) * 2
    )

    # Repeat median-imputed bootstrap
    imputed_repeat = stratified_bootstrap(
        pass_imputed[:, j],
        fail_imputed[:, j],
        BOOTSTRAP_REPETITIONS,
        RANDOM_SEED + int(feature_index) * 2 + 1
    )

    repeated_results.append({
        "Available_Case_Bootstrap_Mean":
            available_repeat["Bootstrap_Mean_Difference"],

        "Available_Case_CI_Lower":
            available_repeat["CI_Lower"],

        "Available_Case_CI_Upper":
            available_repeat["CI_Upper"],

        "Imputed_Bootstrap_Mean":
            imputed_repeat["Bootstrap_Mean_Difference"],

        "Imputed_CI_Lower":
            imputed_repeat["CI_Lower"],

        "Imputed_CI_Upper":
            imputed_repeat["CI_Upper"]
    })

repeat_df = pd.DataFrame(repeated_results)

original_values = bootstrap_df[
    [
        "Available_Case_Bootstrap_Mean",
        "Available_Case_CI_Lower",
        "Available_Case_CI_Upper",
        "Imputed_Bootstrap_Mean",
        "Imputed_CI_Lower",
        "Imputed_CI_Upper"
    ]
].to_numpy()

repeated_values = repeat_df.to_numpy()

reproducibility_verified = np.allclose(
    original_values,
    repeated_values,
    rtol=1e-10,
    atol=1e-10
)

print("BOOTSTRAP REPRODUCIBILITY")
print("------------------------")
print("Original Runs:", len(bootstrap_df))
print("Repeated Runs:", len(repeat_df))
print("Same Random Seeds:", True)
print("Results Match:", reproducibility_verified)


BOOTSTRAP REPRODUCIBILITY
------------------------
Original Runs: 5
Repeated Runs: 5
Same Random Seeds: True
Results Match: True


In [23]:

# [CELL 23] STRATEGY COMPARISON AND FEATURE RANKINGS

robustness_comparison = strategy_comparison.copy()

robustness_comparison["Absolute_Change"] = np.abs(
    robustness_comparison["Median_Imputed_Difference"] -
    robustness_comparison["Available_Case_Difference"]
)

# Compare difference directions
robustness_comparison["Available_Case_Direction"] = np.sign(
    robustness_comparison["Available_Case_Difference"]
)

robustness_comparison["Imputed_Direction"] = np.sign(
    robustness_comparison["Median_Imputed_Difference"]
)

robustness_comparison["Direction_Changed"] = (
    robustness_comparison["Available_Case_Direction"] !=
    robustness_comparison["Imputed_Direction"]
)

# Rank features using absolute observed mean differences
robustness_comparison["Available_Case_Rank"] = (
    robustness_comparison["Available_Case_Difference"]
    .abs()
    .rank(method="min", ascending=False)
    .astype(int)
)

robustness_comparison["Imputed_Rank"] = (
    robustness_comparison["Median_Imputed_Difference"]
    .abs()
    .rank(method="min", ascending=False)
    .astype(int)
)

robustness_comparison["Ranking_Changed"] = (
    robustness_comparison["Available_Case_Rank"] !=
    robustness_comparison["Imputed_Rank"]
)

# Add bootstrap confidence intervals
robustness_comparison = robustness_comparison.merge(
    bootstrap_df[
        [
            "Feature_Index",
            "Available_Case_Bootstrap_Mean",
            "Available_Case_CI_Lower",
            "Available_Case_CI_Upper",
            "Imputed_Bootstrap_Mean",
            "Imputed_CI_Lower",
            "Imputed_CI_Upper"
        ]
    ],
    on="Feature_Index",
    how="left"
)

# Determine whether each confidence interval includes zero
robustness_comparison["Available_Case_CI_Includes_Zero"] = (
    (robustness_comparison["Available_Case_CI_Lower"] <= 0) &
    (robustness_comparison["Available_Case_CI_Upper"] >= 0)
)

robustness_comparison["Imputed_CI_Includes_Zero"] = (
    (robustness_comparison["Imputed_CI_Lower"] <= 0) &
    (robustness_comparison["Imputed_CI_Upper"] >= 0)
)

largest_change = robustness_comparison["Absolute_Change"].max()

changed_sign_count = int(
    robustness_comparison["Direction_Changed"].sum()
)

changed_ranking_count = int(
    robustness_comparison["Ranking_Changed"].sum()
)

print("ROBUSTNESS COMPARISON")
print("---------------------")
display(robustness_comparison.round(6))

print("\nSENSITIVITY SUMMARY")
print("-------------------")
print("Largest Change in Mean Difference:", largest_change)
print("Features with Changed Difference Sign:", changed_sign_count)
print("Features with Changed Ranking:", changed_ranking_count)
print(
    "Available-Case CIs Including Zero:",
    robustness_comparison["Available_Case_CI_Includes_Zero"].sum()
)
print(
    "Imputed CIs Including Zero:",
    robustness_comparison["Imputed_CI_Includes_Zero"].sum()
)
print("Bootstrap Reproducibility Verified:", reproducibility_verified)


ROBUSTNESS COMPARISON
---------------------


,Feature_Index,Available_Case_Difference,Median_Imputed_Difference,Absolute_Change,Available_Case_Direction,Imputed_Direction,Direction_Changed,Available_Case_Rank,Imputed_Rank,Ranking_Changed,Available_Case_Bootstrap_Mean,Available_Case_CI_Lower,Available_Case_CI_Upper,Imputed_Bootstrap_Mean,Imputed_CI_Lower,Imputed_CI_Upper,Available_Case_CI_Includes_Zero,Imputed_CI_Includes_Zero
0,162,-154.061108,-179.583384,25.522276,-1.0,-1.0,False,1,1,False,-154.082686,-1114.133710,920.525405,-233.257260,-1357.082053,1016.906532,True,True
1,417,-0.693796,-0.692646,0.001150,-1.0,-1.0,False,3,3,False,-0.735483,-1.409080,0.019194,-0.690523,-1.392820,0.181072,True,True
2,444,-0.004192,-0.004197,0.000005,-1.0,-1.0,False,5,5,False,-0.003293,-0.027278,0.022741,-0.004911,-0.035354,0.025285,True,True
3,473,0.855168,0.879680,0.024512,1.0,1.0,False,2,2,False,0.979109,-3.803769,6.133184,0.863539,-3.605700,5.440694,True,True
4,522,0.390212,0.390212,0.000000,1.0,1.0,False,4,4,False,0.385296,-0.788383,1.574159,0.416497,-0.777834,1.915534,True,True



SENSITIVITY SUMMARY
-------------------
Largest Change in Mean Difference: 25.52227578322163
Features with Changed Difference Sign: 0
Features with Changed Ranking: 0
Available-Case CIs Including Zero: 5
Imputed CIs Including Zero: 5
Bootstrap Reproducibility Verified: True


In [24]:

# [CELL 24] EXPORT ROBUSTNESS COMPARISON

robustness_comparison.to_csv(
    OUTPUT_DIR / "robustness_comparison.csv",
    index=False
)

print("ROBUSTNESS COMPARISON EXPORT")
print("----------------------------")
print("Exported File:", OUTPUT_DIR / "robustness_comparison.csv")
print("File Exists:",
      (OUTPUT_DIR / "robustness_comparison.csv").exists())
print("Total Features:", len(robustness_comparison))


ROBUSTNESS COMPARISON EXPORT
----------------------------
Exported File: robustness_comparison.csv
File Exists: True
Total Features: 5


In [25]:

# [CELL 25] GENERATE SENSITIVITY REPORT

report_lines = [
    "SENSITIVITY ANALYSIS REPORT",
    "===========================",
    "",
    f"Student: {LAST_NAME}",
    f"Student ID: {STUDENT_ID}",
    f"Random Seed: {RANDOM_SEED}",
    "",
    "ANALYSIS CONFIGURATION",
    "----------------------",
    f"Selected Features: {len(ex3_selected_features)}",
    f"Bootstrap Repetitions: {BOOTSTRAP_REPETITIONS}",
    "Confidence Level: 95%",
    "Difference Calculation: Fail Mean - Pass Mean",
    "",
    "SELECTED FEATURES",
    "-----------------",
    str(ex3_selected_features.tolist()),
    "",
    "SENSITIVITY SUMMARY",
    "-------------------",
    f"Largest Change in Mean Difference: {largest_change:.6f}",
    f"Features with Changed Difference Sign: {changed_sign_count}",
    f"Features with Changed Ranking: {changed_ranking_count}",
    f"Available-Case CIs Including Zero: {int(robustness_comparison['Available_Case_CI_Includes_Zero'].sum())}",
    f"Imputed CIs Including Zero: {int(robustness_comparison['Imputed_CI_Includes_Zero'].sum())}",
    f"Bootstrap Reproducibility Verified: {reproducibility_verified}",
    "",
    "FEATURE COMPARISON",
    "------------------"
]

for _, row in robustness_comparison.iterrows():

    report_lines.append(
        f"Feature {int(row['Feature_Index'])}: "
        f"Available-case difference = "
        f"{row['Available_Case_Difference']:.6f}, "
        f"Imputed difference = "
        f"{row['Median_Imputed_Difference']:.6f}, "
        f"Available-case CI = "
        f"[{row['Available_Case_CI_Lower']:.6f}, "
        f"{row['Available_Case_CI_Upper']:.6f}], "
        f"Imputed CI = "
        f"[{row['Imputed_CI_Lower']:.6f}, "
        f"{row['Imputed_CI_Upper']:.6f}]"
    )

report_lines.extend([
    "",
    "INTERPRETATION",
    "--------------",
    "The two data preparation strategies were compared using",
    "observed Fail-minus-Pass mean differences and stratified",
    "bootstrap confidence intervals.",
    "",
    "Confidence intervals containing zero do not provide clear",
    "evidence of a nonzero mean difference at the 95% level.",
    "",
    "Bootstrap reproducibility was checked by repeating the",
    "analysis with identical random seeds."
])

report_path = OUTPUT_DIR / "sensitivity_report.txt"

report_path.write_text(
    "\n".join(report_lines),
    encoding="utf-8"
)

print("SENSITIVITY REPORT")
print("------------------")
print("Report Created:", report_path.exists())
print("Report Path:", report_path)
print("\n".join(report_lines))


SENSITIVITY REPORT
------------------
Report Created: True
Report Path: sensitivity_report.txt
SENSITIVITY ANALYSIS REPORT

Student: Malitao
Student ID: TUPM-26-1135
Random Seed: 261135

ANALYSIS CONFIGURATION
----------------------
Selected Features: 5
Bootstrap Repetitions: 250
Confidence Level: 95%
Difference Calculation: Fail Mean - Pass Mean

SELECTED FEATURES
-----------------
[162, 417, 444, 473, 522]

SENSITIVITY SUMMARY
-------------------
Largest Change in Mean Difference: 25.522276
Features with Changed Difference Sign: 0
Features with Changed Ranking: 0
Available-Case CIs Including Zero: 5
Imputed CIs Including Zero: 5
Bootstrap Reproducibility Verified: True

FEATURE COMPARISON
------------------
Feature 162: Available-case difference = -154.061108, Imputed difference = -179.583384, Available-case CI = [-1114.133710, 920.525405], Imputed CI = [-1357.082053, 1016.906532]
Feature 417: Available-case difference = -0.693796, Imputed difference = -0.692646, Available-case CI = 

In [26]:

# [CELL 26] FINAL OUTPUT VERIFICATION

required_files = {
    "measurement_quality.csv": 7,
    "balanced_sample.csv": 76,
    "feature_comparison.csv": 5,
    "robustness_comparison.csv": 5,
    "sensitivity_report.txt": None
}

print("FINAL OUTPUT VERIFICATION")
print("=========================")

all_files_verified = True

for filename, expected_rows in required_files.items():

    file_path = OUTPUT_DIR / filename

    exists = file_path.exists()
    readable = False
    row_count = None

    if exists:
        try:
            if filename.endswith(".csv"):
                exported_df = pd.read_csv(file_path)
                row_count = len(exported_df)

                readable = (
                    expected_rows is None or
                    row_count == expected_rows
                )

            else:
                report_content = file_path.read_text(
                    encoding="utf-8"
                )
                readable = len(report_content.strip()) > 0

        except Exception as error:
            print(f"{filename}: ERROR - {error}")

    verified = exists and readable
    all_files_verified &= verified

    print(f"\nFile: {filename}")
    print("Exists:", exists)
    print("Readable / Row Count Correct:", readable)

    if row_count is not None:
        print("Rows:", row_count)

    print("Verified:", verified)

print("\nFINAL STATUS")
print("------------")
print("All Required Files Verified:", all_files_verified)
print("Exercise 1 Completed:", (OUTPUT_DIR / "measurement_quality.csv").exists())
print("Exercise 2 Completed:",
      (OUTPUT_DIR / "balanced_sample.csv").exists()
      and (OUTPUT_DIR / "feature_comparison.csv").exists())
print("Exercise 3 Completed:",
      (OUTPUT_DIR / "robustness_comparison.csv").exists()
      and (OUTPUT_DIR / "sensitivity_report.txt").exists())


FINAL OUTPUT VERIFICATION

File: measurement_quality.csv
Exists: True
Readable / Row Count Correct: True
Rows: 7
Verified: True

File: balanced_sample.csv
Exists: True
Readable / Row Count Correct: True
Rows: 76
Verified: True

File: feature_comparison.csv
Exists: True
Readable / Row Count Correct: True
Rows: 5
Verified: True

File: robustness_comparison.csv
Exists: True
Readable / Row Count Correct: True
Rows: 5
Verified: True

File: sensitivity_report.txt
Exists: True
Readable / Row Count Correct: True
Verified: True

FINAL STATUS
------------
All Required Files Verified: True
Exercise 1 Completed: True
Exercise 2 Completed: True
Exercise 3 Completed: True
